In [101]:
### Time frames - Define the start and end dates of the data that you want to look at (e.g. all 29 years between 1996 and 2025) (ICMECAT only goes to mid-2025 for now)
dataTimeframe = ["1996-01-01 00:00", "2025-01-01 00:00"]
start = dataTimeframe[0] # start date of data
end = dataTimeframe[1] # end date of data
# dataSourcesToUse = ["Wind"]
fileName = "Wind_" + str(start)[0:11] + "-" + str(end)[0:11]+ "InputOutput.csv"

slowCutOff = 400
fastCutOff = 500

# Get imports and functions needed
# from sunpy.net import Fido
# from sunpy.net import attrs as a
# from sunpy.timeseries import TimeSeries 
from sunpy.timeseries import GenericTimeSeries
# from sunpy.time import parse_time
# from sunpy.data import cache
# from sunpy.coordinates import spice, frames
# from sunpy.coordinates import get_horizons_coord, get_body_heliographic_stonyhurst
# from astropy import units as u
# from astropy.time import Time
import matplotlib.pyplot as plt
# import matplotlib.dates as mdates
import numpy as np 
# from datetime import datetime
# from datetime import date, timedelta
import os
# import urllib.request
import pandas as pd
# import datetime as dt
# from scipy.stats import norm
# import seaborn as sns
# import polars as pl
# from scipy import stats
# import scipy.stats as stats
# from scipy.stats import anderson
import calendar
import warnings

import astropy.units as u
import astropy.constants as const
from astropy.visualization import quantity_support
warnings.filterwarnings("ignore")


# Wind

## Read in dataframe

In [102]:
windData = pd.read_csv(fileName, index_col="DateTime")

In [103]:
print(windData)

                     Year  DOY  Hour  Minute  IMF_PTS  PercentInterp  \
DateTime                                                               
1996-01-01 00:00:00  1996    1     0       0      2.0          100.0   
1996-01-01 00:01:00  1996    1     0       1      2.0          100.0   
1996-01-01 00:02:00  1996    1     0       2      1.0          100.0   
1996-01-01 00:03:00  1996    1     0       3      1.0          100.0   
1996-01-01 00:04:00  1996    1     0       4      1.0          100.0   
...                   ...  ...   ...     ...      ...            ...   
2024-12-31 23:55:00  2024  366    23      55      NaN            NaN   
2024-12-31 23:56:00  2024  366    23      56      1.0          100.0   
2024-12-31 23:57:00  2024  366    23      57      1.0          100.0   
2024-12-31 23:58:00  2024  366    23      58      NaN            NaN   
2024-12-31 23:59:00  2024  366    23      59      1.0          100.0   

                     CP/MVFlag  Timeshift  PFN_XGSE  PFN_YGSE  

## Add parameters

In [104]:
# B  = windData["ScalarB"].values * u.nT
Bx = windData["Bx_GSEGSM"].values* u.nT
By = windData["ByGSE"].values * u.nT
Bz = windData["BzGSE"].values* u.nT
# v  = windData["FlowSpeed"].values * u.km / u.s
Vx  = windData["Vx_GSE"].values * u.km / u.s
Vy  = windData["Vy_GSE"].values * u.km / u.s
Vz  = windData["Vz_GSE"].values * u.km / u.s
n  = windData["ProtonDensity"].values * u.cm**-3
T  = windData["Temp"].values * u.K

In [105]:
# total solar wind speed
TotVsw = np.sqrt(Vx**2 + Vy**2 + Vz**2)

# magnitude of magnetic field
BMag = np.sqrt(Bx**2 + By**2 + Bz**2)

#Alfvén speed
vA = BMag / np.sqrt(const.mu0 * n * const.m_p)
vA = vA.to(u.km / u.s)

#Alfvén mach number
Ma = (TotVsw / vA).to(u.dimensionless_unscaled)

#Dynamic pressure
Pdyn = (n * const.m_p * TotVsw**2).to(u.nPa)
# Pdyn_by_hand = 1.6726e-6 * windData["ProtonDensity"] * windData["FlowSpeed"]**2 

#Beta
beta = (n * const.k_B * T / (BMag**2 / (2 * const.mu0))).to(u.dimensionless_unscaled)

#Cone and clock angles
cone = np.arccos(np.clip(Bx / BMag, -1, 1)).to(u.deg)    
clock = np.arctan2(By, Bz).to(u.deg)  

In [106]:
windData["TotVsw"]    = TotVsw.to_value(u.km / u.s)
windData["BMag"]      = BMag.to_value(u.nT)
windData["vA_kms"]    = vA.to_value(u.km / u.s)
windData["Ma"]        = Ma.value
windData["Pdyn_nPa"]  = Pdyn.to_value(u.nPa)
windData["beta"]      = beta.value
windData["cone_deg"]  = cone.to_value(u.deg)
windData["clock_deg"] = clock.to_value(u.deg)

windData.describe().round(2)

,Year,DOY,Hour,Minute,IMF_PTS,PercentInterp,CP/MVFlag,Timeshift,PFN_XGSE,PFN_YGSE,...,DBOT1,DBOT2,TotVsw,BMag,vA_kms,Ma,Pdyn_nPa,beta,cone_deg,clock_deg
count,15213886.00,15213886.00,15213886.00,15213886.00,13523203.00,13523203.00,7986733.00,13521845.00,13523203.00,13523203.00,...,13164279.00,13284596.00,11158718.00,13522112.00,11085113.00,11051533.00,11154510.00,11085113.00,13522112.00,13522351.00
mean,2010.02,183.18,11.50,29.50,5.11,52.57,0.75,2823.03,0.61,0.16,...,60.63,-154.77,424.42,5.58,53.17,10.09,1.89,0.90,90.13,-0.87
std,8.37,105.58,6.92,17.32,4.47,45.31,0.36,1847.45,0.20,0.41,...,525.73,788.84,98.73,3.04,32.03,6.90,1.53,25.79,41.08,100.89
min,1996.00,1.00,0.00,0.00,1.00,0.00,0.00,-10797.00,0.01,-1.00,...,-19999.00,-21225.00,228.71,0.02,0.12,0.46,0.02,0.00,0.00,-179.98
25%,2003.00,92.00,5.00,14.00,2.00,0.00,0.60,1861.00,0.44,-0.12,...,21.00,-4.00,350.56,3.67,34.27,6.81,1.09,0.27,56.22,-91.39
50%,2010.00,184.00,11.00,29.00,4.00,60.00,1.00,2970.00,0.60,0.21,...,47.00,37.00,401.85,4.91,47.57,8.92,1.53,0.52,90.09,-2.34
75%,2017.00,275.00,17.00,44.00,6.00,100.00,1.00,3951.00,0.77,0.48,...,80.00,64.00,477.04,6.66,64.13,11.61,2.22,0.92,124.05,89.76
max,2024.00,366.00,23.00,59.00,408.00,100.00,1.00,10798.00,1.00,0.97,...,20271.00,17188.00,1076.61,70.11,1121.47,3196.61,87.48,73034.12,180.00,180.00


In [107]:
type(pd.to_datetime(windData.index))

pandas.DatetimeIndex

In [108]:
# quantity_support()
# fig, axes = plt.subplots(5, 1, figsize=(9, 9), sharex=True)
# axes[0].plot(windData.index, BMag)
# axes[1].plot(windData.index, TotVsw)
# axes[2].plot(windData.index, Pdyn)
# axes[3].plot(windData.index, beta)
# axes[3].set_yscale("log")
# axes[4].plot(windData.index, Ma)

# for ax, name in zip(axes, ["B", "v", "Pdyn", "beta", "Ma"]):
#     ax.set_title(name, loc="left", fontsize=9)

# fig.tight_layout()

## Add CME column

In [109]:
# print(type(pd.to_datetime(windData.index)))
# print(type(icmes_wind["icme_start_time"][1].round(freq='min')))

# print(windData)

In [110]:
## reading the catalogue
icmes = pd.read_csv("HELIO4CAST_ICMECAT_v23.csv")
icmes_wind = icmes[icmes["sc_insitu"]=="Wind"]
print(icmes_wind)

## just making these times more consistent to be like the df ones
## the .tz_localise to to remove the fact that the catalogue has the datatimes Z at the end.
for column in ["icme_start_time", "mo_start_time", "mo_end_time"]:
    icmes_wind[column] = pd.to_datetime(icmes_wind[column]).dt.tz_localize(None)

## here what we can do is create an ICME column with the same index as the dataframe. 
## then set it all to 0, then look at the time ranges from the catalogue and set 
## this column equal to sheath as 1, cme (magnetic object as 2), and then the rest as 0. you can also just change the start times and end times here etc. 
icme_column = pd.Series(np.int8(0), index=pd.to_datetime(windData.index))
# print(len(icme_column))
print(icme_column)
for start, end in zip(icmes_wind["icme_start_time"], icmes_wind["mo_start_time"]):
    icme_column.loc[start:end] = 1
for start, end in zip(icmes_wind["mo_start_time"], icmes_wind["mo_end_time"]):
    icme_column.loc[start:end] = 2

print(icme_column)

windData["cmes"] = icme_column
print(windData["cmes"])

# you can also change the icme_start_time to be icme_start_time - 12 hours etc. this is just directly from the times from the catalogue.

# then you can do nice things like:
# no_cmes = df.where(df["cmes"] == 0)

# or no_cmes = df[df["cmes"]==0] etc.

# (sorry for code dump!)

      Unnamed: 0.1  Unnamed: 0                    icmecat_id sc_insitu  \
1                1           1  ICME_Wind_MOESTL_20250726_01      Wind   
4                4           4  ICME_Wind_MOESTL_20250703_01      Wind   
9                9           9  ICME_Wind_MOESTL_20250612_01      Wind   
10              10          10  ICME_Wind_MOESTL_20250607_01      Wind   
11              11          11  ICME_Wind_MOESTL_20250604_01      Wind   
...            ...         ...                           ...       ...   
1908          1908        1908    ICME_Wind_NASA_19950405_01      Wind   
1909          1909        1909    ICME_Wind_NASA_19950403_01      Wind   
1910          1910        1910    ICME_Wind_NASA_19950306_01      Wind   
1911          1911        1911    ICME_Wind_NASA_19950304_01      Wind   
1912          1912        1912    ICME_Wind_NASA_19950207_01      Wind   

        icme_start_time      mo_start_time        mo_end_time  \
1     2025-07-26T16:44Z  2025-07-26T16:44Z  20

In [111]:
# icmes_wind["icme_start_time"].iloc[532:540], icmes_wind["mo_start_time"][532:540]
# windData[0:int(365.25*24*60 + 1080)]
# windData[0:int(365.25*24*60 + 1080)]["cmes"].unique()
# icme_column[0:int(365.25*24*60 + 1080)]
# icme_column[0:int(365.25*24*60 + 1080)].unique()

## Add sw type column

In [ ]:
# method 1
# slow_condition = windData["TotVsw"] < slowCutOff
# fast_condition = windData["TotVsw"] > fastCutOff

# windData = windData.assign(sw_type=lambda x: 1 if slow_condition.any() else 0)
# print(windData["TotVsw"][50:100])
# print(windData["sw_type"][50:100])

# method 2

# windData['sw_type'] = 0
# windData.loc[windData['TotVsw']  < slowCutOff] = 1
# # windData.loc[windData['TotVsw']  > fastCutOff] = 2
# print(windData["TotVsw"][50:100])
# print(windData["sw_type"][50:100])

sw_type_column = pd.Series(np.int8(0), index=pd.to_datetime(windData.index))
# print(len(icme_column))
# print(icme_column)
# for start, end in zip(icmes_wind["icme_start_time"], icmes_wind["mo_start_time"]):
#     icme_column.loc[start:end] = 1
# for start, end in zip(icmes_wind["mo_start_time"], icmes_wind["mo_end_time"]):
#     icme_column.loc[start:end] = 2

# windData['sw_type'] = 0
sw_type_column.loc[windData['TotVsw']  < slowCutOff] = 1
# windData.loc[windData['TotVsw']  > fastCutOff] = 2
# print(windData["TotVsw"][50:100])
# print(windData["sw_type"][50:100])

IndexingError: Unalignable boolean Series provided as indexer (index of the boolean Series and of the indexed object do not match).

In [ ]:
sw_type_column = pd.Series(np.int8(0), index=pd.to_datetime(windData.index))

# print(windData.columns)
# for v_sw in windData["TotVsw"]:
#     # print(v_sw)
#     sw_type_column.loc[v_sw < slowCutOff] = "1"
# for v_sw in windData["TotVsw"]:
#     sw_type_column.loc[v_sw > fastCutOff] = "2"

print(sw_type_column[50:100])

sw_type_column = sw_type_column.mask(windData['TotVsw'] < slowCutOff, other = 1)
# print(sw_type_column.values.unique)


# print(sw_type_column.unique)

# print(windData["sw_type"].unique)
# ## here what we can do is create an ICME column with the same index as the dataframe. 
# ## then set it all to 0, then look at the time ranges from the catalogue and set 
# ## this column equal to sheath as 1, cme (magnetic object as 2), and then the rest as 0. you can also just change the start times and end times here etc. 
# icme_column = pd.Series(np.int8(0), index=pd.to_datetime(windData.index))
# # print(len(icme_column))
# print(icme_column)
# for start, end in zip(icmes_wind["icme_start_time"], icmes_wind["mo_start_time"]):
#     icme_column.loc[start:end] = 1
# for start, end in zip(icmes_wind["mo_start_time"], icmes_wind["mo_end_time"]):
#     icme_column.loc[start:end] = 2

# print(icme_column)

# windData["sw_type"] = sw_type_column
# print(windData["sw_type"])

# you can also change the icme_start_time to be icme_start_time - 12 hours etc. this is just directly from the times from the catalogue.

# then you can do nice things like:
# no_cmes = df.where(df["cmes"] == 0)

# or no_cmes = df[df["cmes"]==0] etc.

DateTime
1996-01-01 00:50:00    0
1996-01-01 00:51:00    0
1996-01-01 00:52:00    0
1996-01-01 00:53:00    0
1996-01-01 00:54:00    0
1996-01-01 00:55:00    0
1996-01-01 00:56:00    0
1996-01-01 00:57:00    0
1996-01-01 00:58:00    0
1996-01-01 00:59:00    0
1996-01-01 01:00:00    0
1996-01-01 01:01:00    0
1996-01-01 01:02:00    0
1996-01-01 01:03:00    0
1996-01-01 01:04:00    0
1996-01-01 01:05:00    0
1996-01-01 01:06:00    0
1996-01-01 01:07:00    0
1996-01-01 01:08:00    0
1996-01-01 01:09:00    0
1996-01-01 01:10:00    0
1996-01-01 01:11:00    0
1996-01-01 01:12:00    0
1996-01-01 01:13:00    0
1996-01-01 01:14:00    0
1996-01-01 01:15:00    0
1996-01-01 01:16:00    0
1996-01-01 01:17:00    0
1996-01-01 01:18:00    0
1996-01-01 01:19:00    0
1996-01-01 01:20:00    0
1996-01-01 01:21:00    0
1996-01-01 01:22:00    0
1996-01-01 01:23:00    0
1996-01-01 01:24:00    0
1996-01-01 01:25:00    0
1996-01-01 01:26:00    0
1996-01-01 01:27:00    0
1996-01-01 01:28:00    0
1996-01-01 01:29

TypeError: NDFrame.mask() takes from 2 to 3 positional arguments but 4 were given

# Save to csv

In [91]:
s = 50
e = s + 50
print(windData["TotVsw"][s:e])
print(sw_type_column[s:e])

DateTime
1996-01-01 00:50:00    427.700748
1996-01-01 00:51:00    409.336243
1996-01-01 00:52:00    396.237921
1996-01-01 00:53:00    396.526052
1996-01-01 00:54:00    399.031603
1996-01-01 00:55:00    397.662030
1996-01-01 00:56:00    396.382719
1996-01-01 00:57:00    399.292700
1996-01-01 00:58:00    401.140636
1996-01-01 00:59:00    401.140636
1996-01-01 01:00:00    395.272741
1996-01-01 01:01:00    395.272741
1996-01-01 01:02:00    397.679180
1996-01-01 01:03:00    397.679180
1996-01-01 01:04:00    396.518259
1996-01-01 01:05:00    396.518259
1996-01-01 01:06:00    395.185058
1996-01-01 01:07:00    395.185058
1996-01-01 01:08:00    396.037435
1996-01-01 01:09:00    396.037435
1996-01-01 01:10:00    397.487622
1996-01-01 01:11:00    397.136727
1996-01-01 01:12:00    396.363079
1996-01-01 01:13:00    395.929426
1996-01-01 01:14:00    395.927708
1996-01-01 01:15:00    394.222856
1996-01-01 01:16:00    395.711991
1996-01-01 01:17:00    396.303078
1996-01-01 01:18:00    396.462419
1996-

In [88]:
sw_type_column = sw_type_column.mask(windData['TotVsw'] > fastCutOff, 2)


In [89]:
s = 50
e = s + 50
print(windData["TotVsw"][s:e])
print(sw_type_column[s:e])

DateTime
1996-01-01 00:50:00    427.700748
1996-01-01 00:51:00    409.336243
1996-01-01 00:52:00    396.237921
1996-01-01 00:53:00    396.526052
1996-01-01 00:54:00    399.031603
1996-01-01 00:55:00    397.662030
1996-01-01 00:56:00    396.382719
1996-01-01 00:57:00    399.292700
1996-01-01 00:58:00    401.140636
1996-01-01 00:59:00    401.140636
1996-01-01 01:00:00    395.272741
1996-01-01 01:01:00    395.272741
1996-01-01 01:02:00    397.679180
1996-01-01 01:03:00    397.679180
1996-01-01 01:04:00    396.518259
1996-01-01 01:05:00    396.518259
1996-01-01 01:06:00    395.185058
1996-01-01 01:07:00    395.185058
1996-01-01 01:08:00    396.037435
1996-01-01 01:09:00    396.037435
1996-01-01 01:10:00    397.487622
1996-01-01 01:11:00    397.136727
1996-01-01 01:12:00    396.363079
1996-01-01 01:13:00    395.929426
1996-01-01 01:14:00    395.927708
1996-01-01 01:15:00    394.222856
1996-01-01 01:16:00    395.711991
1996-01-01 01:17:00    396.303078
1996-01-01 01:18:00    396.462419
1996-

## Save to csv file

In [ ]:
# windDataConcat.to_csv("Wind_" + str(start)[0:10] + "-" + str(end)[0:10]+ "InputOutput.csv")